In [28]:
import pandas as pd

df = pd.read_csv(r'C:\유비온프로젝트2\corporate-bankruptcy\데이터수집\재무,비재무지표(TS2000)\산업별_데이터셋\M25_오락_문화_개인서비스업.csv', encoding='utf-8-sig')

selected_cols = [
    '회사명', '거래소코드', '회계년도', '사업자등록번호',
    '통계청 한국표준산업분류 코드 11차(중분류)',
    '자산총계(요약)(백만원)', '유동자산(요약)(백만원)',
    '현금 및 현금성자산(요약)(백만원)', '매출채권(요약)(백만원)',
    '재고자산(요약)(백만원)', '비유동자산(요약)(백만원)', '유형자산(요약)(백만원)',
    '부채총계(요약)(백만원)', '유동부채(요약)(백만원)', '매입채무(요약)(백만원)',
    '단기차입금(요약)(백만원)', '유동성장기부채(요약)(백만원)', '사채(요약)(백만원)',
    '장기차입금(요약)(백만원)', '비유동부채(요약)(백만원)', '자본총계(요약)(백만원)',
    '자본금(요약)(백만원)', '이익잉여금(요약)(백만원)', '매출액(요약)(백만원)',
    '매출원가(요약)(백만원)', '매출총이익(요약)(백만원)', '판매비와 관리비(요약)(백만원)',
    '급료', '감가상각비', '영업이익(요약)(백만원)', '이자비용(요약)(백만원)',
    '당기순이익(요약)(백만원)', '법인세비용차감전(계속사업)손익(요약)(백만원)',
    '(계속사업손익)법인세비용(요약)(백만원)', '주당순이익(요약)(원)',
    '영업활동으로 인한 현금흐름(요약)(백만원)', '투자활동으로 인한 현금유출액(요약)(백만원)',
]

df = df[selected_cols].copy()

# 회계년도 앞 4자리를 연도로 파싱
df['_year'] = df['회계년도'].str[:4].astype(int)

# 사업자등록번호별 마지막 연도의 회사명·거래소코드 추출
latest = (
    df.sort_values('_year')
    .groupby('사업자등록번호')
    .last()[['회사명', '거래소코드']]
    .reset_index()
    .rename(columns={'회사명': '최신_회사명', '거래소코드': '최신_거래소코드'})
)

# 병합 후 교체
df = df.merge(latest, on='사업자등록번호', how='left')
df['회사명'] = df['최신_회사명']
df['거래소코드'] = df['최신_거래소코드']
df.drop(columns=['최신_회사명', '최신_거래소코드', '_year'], inplace=True)

# 컬럼명 변경
df.rename(columns={'통계청 한국표준산업분류 코드 11차(중분류)': '한국표준산업분류코드'}, inplace=True)

df.to_csv('M25_오락_문화_개인서비스업_정제.csv', index=False, encoding='utf-8-sig')